## Data Exploration

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import re
import numpy as np

pd.set_option('display.max_colwidth', None)

ROOT = Path.cwd()
while not (ROOT / ".git").exists():
    ROOT = ROOT.parent

DATA = ROOT / "data" / "prb_articles_labeled_Jan2016-Jun2026_duplicate-free.json"
DATA_SAMPLE = ROOT / "data" / "prb_articles_labeled_Jan2016-Jun2026_duplicate-free_sample.json"

MAPPER_MODULE_PATH = str(ROOT / "src" / "data")

#For importing concept_id_to_physh_name() from preprocessing.py
if MAPPER_MODULE_PATH not in sys.path:
    sys.path.append(MAPPER_MODULE_PATH)

data_to_load = DATA if DATA.exists() else DATA_SAMPLE
df = pd.read_json(data_to_load)

print(f"Loaded data from {data_to_load.name}")
df.info()

### Mapping PhySH tags based on concept IDs

In [ ]:
from preprocessing import concept_id_to_physh_name

concept_id_map = concept_id_to_physh_name()

In [ ]:
#Applying it to the concept IDs in the dataset and mapping it to PhySH tag names.
df["physh_names"]=df["physh"].apply(
    lambda uuid_list: [
        concept_id_map.get(uuid,None) for uuid in uuid_list
    ]
)

### Statistics on the full dataset

In [ ]:
df.info()

In [ ]:
latex_pattern = re.compile(r'\$|\\[a-zA-Z]+|\{|\}')
latex_present = df['abstract'].apply(lambda x : bool(latex_pattern.search(x)))

In [ ]:
print(f"Abstracts with latex: {latex_present.sum()}")
df[latex_present]['abstract']

### PhySH tags exploration

In [ ]:
#len(set(tag for tags in df['physh'] for tag in tags))
#len(set(np.concatenate(list(df['physh']))))
df['physh_names'].explode().nunique()

In [ ]:
df_label_counts = df['physh'].explode().value_counts().to_frame(name='counts')

In [ ]:
df_label_counts.sort_values(by='counts', ascending=True)

In [ ]:
#len(list(df_label_counts[df_label_counts['counts']<50].index))
df_label_counts[df_label_counts['counts']<50].shape

In [ ]:
#Labels that occur less than 50 times in the data
labels_low_freq = list(df_label_counts[df_label_counts['counts']<50].index)
#Remove the low frequency labels
df['physh'] = df['physh'].apply(lambda x: [item for item in x if item not in labels_low_freq])